# SpinnyBall Tutorial: Gyroscopic Mass-Stream Digital Twin

This tutorial introduces the SpinnyBall physics simulation framework for closed-loop shepherded gyroscopic mass-stream dynamics. You'll learn:

- **Core physics**: Full 3D rigid-body dynamics with explicit gyroscopic coupling
- **Rigid body simulation**: Using the `RigidBody` class with quaternion attitudes
- **Multi-body streams**: Event-driven magnetic capture/release at S-Nodes
- **Control systems**: Model-predictive control (MPC) for libration damping
- **Monte-Carlo analysis**: Uncertainty quantification and cascade risk assessment

## Prerequisites

Install dependencies:
```bash
poetry install
poetry install --extras mpc --extras ml --extras monte-carlo
```

## 1. Rigid Body Dynamics with Gyroscopic Coupling

The core physics implements the corrected Euler rotational dynamics equation:

$$\mathbf{I} \dot{\boldsymbol{\omega}} + \boldsymbol{\omega} \times (\mathbf{I} \boldsymbol{\omega}) = \boldsymbol{\tau}_\text{mag} + \boldsymbol{\tau}_\text{grav} + \boldsymbol{\tau}_\text{solar} + \boldsymbol{\tau}_\text{tether}$$

The skew-symmetric term $\boldsymbol{\omega} \times (\mathbf{I} \boldsymbol{\omega})$ **is** the gyroscopic coupling that produces precession and libration.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dynamics.rigid_body import RigidBody

# Create a spin-stabilized packet (Sovereign Bean)
# Typical parameters: ~50g sphere, ~50 krpm spin
mass = 0.05  # kg
radius = 0.02  # m
I_sphere = (2.0/5.0) * mass * radius**2
# Add slight asymmetry to induce precession
I = np.diag([I_sphere, I_sphere * 1.1, I_sphere * 0.9])

# Initial spin about intermediate axis (unstable, induces precession)
omega0 = np.array([0.0, 100.0, 10.0])  # rad/s (~950 rpm)
q0 = np.array([0.0, 0.0, 0.0, 1.0])  # Identity quaternion

packet = RigidBody(mass, I, angular_velocity=omega0, quaternion=q0)

print(f"Packet mass: {mass} kg")
print(f"Packet radius: {radius*1000:.1f} mm")
print(f"Initial angular velocity: {omega0} rad/s")
print(f"Angular momentum: {packet.angular_momentum}")
print(f"Rotational energy: {packet.rotational_energy:.6f} J")

### Torque-Free Precession

Under zero torque, an asymmetric rigid body exhibits precession. The gyroscopic coupling term causes the angular momentum to precess around the principal axis.

In [ ]:
def zero_torque(t, state):
    """Zero external torque for torque-free precession."""
    return np.array([0.0, 0.0, 0.0])

# Integrate for several precession periods
result = packet.integrate(
    t_span=(0.0, 10.0),
    torques=zero_torque,
    method="RK45",
    rtol=1e-10,
    atol=1e-12,
)

print(f"Integration complete: {len(result['t'])} time steps")
print(f"Final angular velocity: {packet.angular_velocity} rad/s")
print(f"Final angular momentum: {packet.angular_momentum}")

In [ ]:
# Plot angular velocity evolution
fig, axes = plt.subplots(2, 1, figsize=(10, 8))

# Angular velocity components
axes[0].plot(result['t'], result['state'][4, :], label='$\omega_x$')
axes[0].plot(result['t'], result['state'][5, :], label='$\omega_y$')
axes[0].plot(result['t'], result['state'][6, :], label='$\omega_z$')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Angular velocity (rad/s)')
axes[0].set_title('Angular Velocity Evolution (Torque-Free Precession)')
axes[0].legend()
axes[0].grid(True)

# Quaternion components
axes[1].plot(result['t'], result['state'][0, :], label='$q_x$')
axes[1].plot(result['t'], result['state'][1, :], label='$q_y$')
axes[1].plot(result['t'], result['state'][2, :], label='$q_z$')
axes[1].plot(result['t'], result['state'][3, :], label='$q_w$')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Quaternion components')
axes[1].set_title('Quaternion Attitude Evolution')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

### Angular Momentum Conservation

The physics gate test verifies angular momentum is conserved to 1e-9 relative tolerance. This confirms the gyroscopic coupling term is correctly implemented.

In [ ]:
# Compute angular momentum throughout trajectory
L_history = []
for i in range(len(result['t'])):
    omega = result['state'][4:7, i]
    L = I @ omega
    L_history.append(np.linalg.norm(L))

L_history = np.array(L_history)
L_initial = L_history[0]
L_final = L_history[-1]
relative_error = np.abs(L_final - L_initial) / L_initial

print(f"Initial angular momentum magnitude: {L_initial:.10f}")
print(f"Final angular momentum magnitude: {L_final:.10f}")
print(f"Relative error: {relative_error:.2e}")
print(f"Physics gate (1e-9): {'PASS' if relative_error < 1e-9 else 'FAIL'}")

# Plot angular momentum conservation
plt.figure(figsize=(10, 4))
plt.plot(result['t'], L_history)
plt.xlabel('Time (s)')
plt.ylabel('Angular momentum magnitude (kg·m²/s)')
plt.title('Angular Momentum Conservation')
plt.grid(True)
plt.show()

## 2. Multi-Body Packet Streams

The multi-body module simulates N=5-20 packets with event-driven magnetic capture/release at sparse S-Nodes (Shepherding Nodes).

In [ ]:
from dynamics.multi_body import MultiBodyStream, Packet, SNode, PacketState

# Create packets
packets = []
for i in range(5):
    mass = 0.05
    I = np.diag([0.0001, 0.00011, 0.00009])
    omega = np.array([0.0, 100.0, 0.0])
    position = np.array([i * 1.0, 0.0, 0.0])
    velocity = np.array([10.0, 0.0, 0.0])
    
    body = RigidBody(mass, I, position=position, velocity=velocity, angular_velocity=omega)
    packet = Packet(id=i, body=body, eta_ind=0.90)
    packets.append(packet)

# Create S-Nodes (Shepherding Nodes)
nodes = [
    SNode(position=np.array([2.5, 0.0, 0.0]), capture_radius=1.0, max_packets=3),
    SNode(position=np.array([5.0, 0.0, 0.0]), capture_radius=1.0, max_packets=3),
]

# Create multi-body stream
stream = MultiBodyStream(packets, nodes, stream_velocity=10.0)

print(f"Created {len(packets)} packets")
print(f"Created {len(nodes)} S-Nodes")
print(f"Stream velocity: {stream.stream_velocity} m/s")

### Event-Driven Capture/Release

Packets are automatically captured when entering an S-Node's capture radius, provided the node has capacity and the packet meets efficiency constraints (η_ind ≥ 0.82).

In [ ]:
# Define torque function (zero for this demo)
def zero_torques(packet_id, t, state):
    return np.array([0.0, 0.0, 0.0])

# Integrate stream
dt = 0.01
metrics_history = []

for step in range(100):
    result = stream.integrate(dt, zero_torques)
    metrics = stream.get_stream_metrics()
    metrics_history.append(metrics)
    
    if step % 20 == 0:
        print(f"Step {step}: {metrics['free_packets']} free, {metrics['captured_packets']} captured")

print(f"\nFinal state: {metrics['free_packets']} free, {metrics['captured_packets']} captured")

## 3. Model-Predictive Control (MPC)

The MPC controller solves optimal control problems with horizon N=10 to minimize libration energy and spacing deviation subject to stress and efficiency constraints.

**Note**: Requires CasADi installation (`poetry install --extras mpc`).

In [ ]:
try:
    from control.mpc_controller import create_mpc_controller
    
    # Create MPC controller
    controller = create_mpc_controller(
        horizon=10,
        dt=0.01,
        libration_weight=1.0,
        spacing_weight=0.5,
        control_weight=0.1,
    )
    
    print("MPC controller created successfully")
    print(f"Horizon: {controller.horizon}")
    print(f"Time step: {controller.dt} s")
    
except ImportError as e:
    print(f"MPC not available: {e}")
    print("Install with: poetry install --extras mpc")

### MPC Solve Time Benchmark

The target is ≤30 ms solve time for real-time control. This benchmark measures actual latency.

In [ ]:
try:
    from control.mpc_controller import verify_mpc_latency
    
    # Run latency benchmark
    benchmark = verify_mpc_latency(controller, n_trials=10)
    
    print(f"\n=== MPC Latency Benchmark ===")
    print(f"Mean solve time: {benchmark['mean_ms']:.2f} ms")
    print(f"Std deviation: {benchmark['std_ms']:.2f} ms")
    print(f"Max solve time: {benchmark['max_ms']:.2f} ms")
    print(f"Target (30 ms): {'PASS' if benchmark['meets_target'] else 'FAIL'}")
    
except NameError:
    print("MPC controller not available - skipping benchmark")

## 4. Monte-Carlo Uncertainty Quantification

The Monte-Carlo framework runs ≥10³ realizations with debris/thermal transients to assess cascade risk and verify pass/fail gates.

Pass/fail gates:
- η_ind ≥ 0.82 (induction efficiency)
- σ ≤ 1.2 GPa (centrifugal stress)
- k_eff ≥ 6,000 N/m (effective stiffness)
- Cascade probability < 10⁻⁶

In [ ]:
from monte_carlo.pass_fail_gates import PassFailGate, GateDirection

# Define pass/fail gates
gates = [
    PassFailGate(name="eta_ind", threshold=0.82, direction=GateDirection.GTE),
    PassFailGate(name="centrifugal_stress", threshold=1.2e9, direction=GateDirection.LTE),
    PassFailGate(name="k_eff", threshold=6000.0, direction=GateDirection.GTE),
]

# Test gate evaluation
test_metrics = {
    "eta_ind": 0.85,
    "centrifugal_stress": 1.1e9,
    "k_eff": 6500.0,
}

print("=== Gate Evaluation ===")
for gate in gates:
    result = gate.evaluate(test_metrics.get(gate.name, 0))
    print(f"{gate.name}: {result.status} (value={test_metrics.get(gate.name, 0)}, threshold={gate.threshold})")

## 5. Running Full Simulations

The project includes several simulation scripts for different use cases:

- `lob_scaling.py`: Multi-node lattice scaling and survivability
- `sgms_anchor_v1.py`: Dynamic anchor stability with parameter sweeps
- `sgms_anchor_suite.py`: Config-driven experiment pipeline

Example usage:
```bash
# Run lattice simulation
python lob_scaling.py --nodes 20

# Run anchor simulation with audit mode
python sgms_anchor_v1.py --audit

# Run config-driven experiment suite
python sgms_anchor_suite.py --repro --output artifacts
```

## 6. Physics Gate Tests

Run the physics gate tests to verify fundamental conservation laws:

```bash
# Run all physics gate tests
pytest tests/test_rigid_body.py -v

# Run specific angular momentum conservation test
pytest tests/test_rigid_body.py::TestTorqueFreePrecession::test_angular_momentum_conservation -v
```

## 7. MuJoCo Oracle Validation

For high-fidelity 6-DoF validation, MuJoCo serves as the ground-truth oracle:

```bash
# Install MuJoCo
poetry install --extras validation

# Run oracle validation
python sgms_anchor_mujoco.py
```

## Summary

This tutorial covered:

1. **Rigid body dynamics** with explicit gyroscopic coupling
2. **Angular momentum conservation** verification (physics gate)
3. **Multi-body packet streams** with event-driven capture/release
4. **Model-predictive control** for libration damping
5. **Monte-Carlo uncertainty quantification** with pass/fail gates

### Next Steps

- Explore the `dynamics/` module for advanced features
- Run the full experiment suite with `sgms_anchor_suite.py`
- Check the digital twin dashboard (`digital_twin.html`)
- Read the blueprint alignment document (`background/ideal_blueprint_alignment.md`)

### References

- Goldstein, Classical Mechanics (3rd ed.), Chapter 4
- Landau & Lifshitz, Mechanics, §35
- Hughes, Spacecraft Attitude Dynamics, Chapter 2